#Chat Metrics

##Data Collection

In [1]:
import pandas            as pd
import numpy             as np
import matplotlib.pyplot as plt
import seaborn           as sns
import graphviz

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
  accuracy_score,
  precision_score,
  recall_score,
  f1_score,
  roc_auc_score,
  classification_report
)
from sklearn import tree

#Read in from hugging face
df = pd.read_parquet('https://huggingface.co/datasets/ddds-Capstone/Datasets/resolve/main/chat_metrics.parquet')

##Processing

In [2]:
features = [

  #Time
  'day_sin',
  'day_cos',
  'hour_sin',
  'hour_cos',
  'month_sin',
  'month_cos',

  #Performance
  'processing_time_seconds',
  'total_tokens',
  'input_tokens',
  'output_tokens',
  'model_calls',
  'tool_calls_count',

  #Q&A
  'question_length',
  'answer_length',

  #Maps
  'has_geolocation',

  #Primary Category
  'primary_category_greetings',
  'primary_category_sunport_amenities',
  'primary_category_navigation',
  'primary_category_airline_logistics',
  'primary_category_general_info',

  #Selected Agent
  'selected_agentreporter',
  'selected_agentplanner',
  'selected_agentlocation',
  'selected_agentlocation_fallback_to_reporter',
  'selected_agentbroad_search_synthesis',
  'selected_agentbroad_search_passthrough',

  #Sentiment
  'question_sentiment',
  'answer_sentiment'
]

X = df[features]
y = df['satisfaction']

###Train/test split

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
  X,
  y,
  test_size=0.20,
  random_state=42,
  stratify=y #Maintains roughly same proportion
)

###Scale

In [4]:
#Split data
X_train, X_test, y_train, y_test = train_test_split(
  X,
  y,
  test_size=0.20,
  random_state=42
)

#Initialize scaler
scaler = StandardScaler()

#Fit and transform training data
X_train = scaler.fit_transform(X_train)

#Transform (DO NOT fit again)
X_test = scaler.transform(X_test)

X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

#Split scaled data
X_train, X_test, y_train, y_test = train_test_split(
  X_scaled,
  y,
  test_size=0.20,
  random_state=42
)

###Logistic Regression

In [5]:
#Create model
lr = LogisticRegression()

#Fit
lr.fit(X_train, y_train)

#Make predictions
y_pred_lr = lr.predict(X_test)

#Probability
y_prob_lr = lr.predict_proba(X_test)[:,1]

#Evaluate
print('Accuracy:', accuracy_score(y_test, y_pred_lr))
print('Precision:', precision_score(y_test, y_pred_lr))
print('Recall:', recall_score(y_test, y_pred_lr))
print('F1:', f1_score(y_test, y_pred_lr))
print('ROC-AUC:', roc_auc_score(y_test, y_prob_lr))

print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr))

Accuracy: 0.7701149425287356
Precision: 0.8529411764705882
Recall: 0.6590909090909091
F1: 0.7435897435897436
ROC-AUC: 0.7669133192389007

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.88      0.79        43
           1       0.85      0.66      0.74        44

    accuracy                           0.77        87
   macro avg       0.78      0.77      0.77        87
weighted avg       0.79      0.77      0.77        87



In [6]:
#Cross Validation
#Use X_scaled to fix convergence issues and 'accuracy' for classification
results = cross_val_score(
  lr,
  X_scaled,
  y,
  scoring='accuracy',
  cv=10
)

#Accuracy across  10 folds
cv_accuracy = results.mean()
print('CV Accuracy:', cv_accuracy)

CV Accuracy: 0.7178118393234671


###Decision Tree

In [7]:
#Model
dt = DecisionTreeClassifier(
  max_depth=3,
  random_state=42
)

#Fit
dt.fit(X_train, y_train)

#Predict
y_pred_dt = dt.predict(X_test)

#Probability
y_prob_dt = dt.predict_proba(X_test)[:,1]

print('Accuracy:', accuracy_score(y_test, y_pred_dt))
print('Precision:', precision_score(y_test, y_pred_dt))
print('Recall:', recall_score(y_test, y_pred_dt))
print('F1:', f1_score(y_test, y_pred_dt))
print('ROC-AUC:', roc_auc_score(y_test, y_prob_dt))

print('\nClassification Report:')
print(classification_report(y_test, y_pred_dt))

Accuracy: 0.6551724137931034
Precision: 0.6346153846153846
Recall: 0.75
F1: 0.6875
ROC-AUC: 0.7404862579281184

Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.56      0.62        43
           1       0.63      0.75      0.69        44

    accuracy                           0.66        87
   macro avg       0.66      0.65      0.65        87
weighted avg       0.66      0.66      0.65        87



In [8]:
#Cross Validation
scores = cross_val_score(
  dt,
  X_train,
  y_train,
  cv=5,
  scoring='accuracy'
)

print('CV Accuracy Scores for each fold:', scores)
print('Mean CV Accuracy:', scores.mean())

CV Accuracy Scores for each fold: [0.60869565 0.5942029  0.71014493 0.69565217 0.72463768]
Mean CV Accuracy: 0.6666666666666667


In [9]:
#Find depth
depths = range(1,11)

results = []

for depth in depths:

  tree_model = DecisionTreeClassifier(
    max_depth=depth,
    random_state=42
  )

  scores = cross_val_score(
    tree_model,
    X_train,
    y_train,
    cv=5,
    scoring = 'accuracy'
  )

  results.append({
    'max_depth': depth,
    'accuracy': scores.mean()
  })

results_df = pd.DataFrame(results)
results_df.sort_values(by='accuracy', ascending=False)

,max_depth,accuracy
3,4,0.684058
1,2,0.681159
5,6,0.672464
4,5,0.672464
7,8,0.672464
2,3,0.666667
0,1,0.663768
6,7,0.652174
8,9,0.652174
9,10,0.649275


In [10]:
#Final
dt = DecisionTreeClassifier(
  max_depth=4,
  random_state=42
)

dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:,1]

print('Accuracy:', accuracy_score(y_test, y_pred_dt))
print('Precision:', precision_score(y_test, y_pred_dt))
print('Recall:', recall_score(y_test, y_pred_dt))
print('F1:', f1_score(y_test, y_pred_dt))
print('ROC-AUC:', roc_auc_score(y_test, y_prob_dt))

print('\nClassification Report:')
print(classification_report(y_test, y_pred_dt))

Accuracy: 0.6206896551724138
Precision: 0.6341463414634146
Recall: 0.5909090909090909
F1: 0.611764705882353
ROC-AUC: 0.6625264270613108

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.65      0.63        43
           1       0.63      0.59      0.61        44

    accuracy                           0.62        87
   macro avg       0.62      0.62      0.62        87
weighted avg       0.62      0.62      0.62        87



In [11]:
#Feature importance
dt_importances = pd.Series(
  dt.feature_importances_,
  index=features
).sort_values(ascending=False)

dt_importances.head(10)

,0
output_tokens,0.406096
answer_sentiment,0.158727
model_calls,0.119771
month_cos,0.106172
month_sin,0.079042
question_sentiment,0.057773
input_tokens,0.044817
primary_category_general_info,0.020387
selected_agentplanner,0.007216
hour_sin,0.000000


###Random Forest

In [12]:
rf = RandomForestClassifier(
  n_estimators=100, #Build 100 individual trees
  max_depth=3,      #Each tree grows max of 3 depths
  random_state=42,  #Makes results reproducable
  n_jobs=-1         #Uses all available CPU cores
)

#Fit
rf.fit(X_train, y_train)

#Predict
y_pred_rf = rf.predict(X_test)

#Probability
y_prob_rf = rf.predict_proba(X_test)[:,1]

print('Accuracy:', accuracy_score(y_test, y_pred_rf))
print('Precision:', precision_score(y_test, y_pred_rf))
print('Recall:', recall_score(y_test, y_pred_rf))
print('F1:', f1_score(y_test, y_pred_rf))
print('ROC-AUC:', roc_auc_score(y_test, y_prob_rf))

print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf))

Accuracy: 0.6666666666666666
Precision: 0.6923076923076923
Recall: 0.6136363636363636
F1: 0.6506024096385542
ROC-AUC: 0.791754756871036

Classification Report:
              precision    recall  f1-score   support

           0       0.65      0.72      0.68        43
           1       0.69      0.61      0.65        44

    accuracy                           0.67        87
   macro avg       0.67      0.67      0.67        87
weighted avg       0.67      0.67      0.67        87



In [13]:
#Tune max_depth
results = []

for depth in depths:

  rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=depth,
    random_state=42,
    n_jobs=-1
  )

  scores = cross_val_score(
    rf,
    X_train,
    y_train,
    cv=5,
    scoring = 'accuracy'
  )

  results.append({
    'max_depth': depth,
    'accuracy': scores.mean()
  })

results_df = pd.DataFrame(results)
results_df.sort_values(by='accuracy', ascending=False)

,max_depth,accuracy
4,5,0.759420
9,10,0.750725
7,8,0.747826
8,9,0.744928
3,4,0.742029
6,7,0.742029
5,6,0.736232
2,3,0.718841
1,2,0.707246
0,1,0.672464


In [ ]:
#Tune n_estimators
n_estimators_values = [50, 100, 200, 300, 500]

results = []

for n in n_estimators_values:

  forest = RandomForestClassifier(
    n_estimators=n,
    max_depth=5,
    random_state=42,
    n_jobs=-1
  )

  scores = cross_val_score(
    forest,
    X_train,
    y_train,
    cv=5,
    scoring = 'accuracy'
  )

  rmse_scores = -scores

  results.append({
    'n_estimators': n,
    'accuracy': scores.mean()
  })

results_df = pd.DataFrame(results)
results_df.sort_values(by='accuracy', ascending=False)

In [ ]:
#Final
rf = RandomForestClassifier(
  n_estimators=100,
  max_depth=5,
  random_state=42,
  n_jobs=-1
)

#Fit
rf.fit(X_train, y_train)

#Predict
y_pred_rf = rf.predict(X_test)

#Probability
y_prob_rf = rf.predict_proba(X_test)[:,1]

print('Accuracy:', accuracy_score(y_test, y_pred_rf))
print('Precision:', precision_score(y_test, y_pred_rf))
print('Recall:', recall_score(y_test, y_pred_rf))
print('F1:', f1_score(y_test, y_pred_rf))
print('ROC-AUC:', roc_auc_score(y_test, y_prob_rf))

print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf))

###XGBoost

In [ ]:
#Create baseline model
xgb = XGBClassifier(
  n_estimators=100,   #Boosting rounds/trees
  learning_rate=0.05, #Controls how much each new tree contributest to model
  max_depth=3,        #Max deptho or each tree
  random_state=42,
  objective = 'binary:logistic', # Tells XGBoost to do binary classification
  eval_metric='logloss'
)

#Fit
xgb.fit(X_train, y_train)

#Predict
y_pred_xgb = xgb.predict(X_test)

#Probability
y_prob_xgb = xgb.predict_proba(X_test)[:,1]

print('Accuracy:', accuracy_score(y_test, y_pred_xgb))
print('Precision:', precision_score(y_test, y_pred_xgb))
print('Recall:', recall_score(y_test, y_pred_xgb))
print('F1:', f1_score(y_test, y_pred_xgb))
print('ROC-AUC:', roc_auc_score(y_test, y_prob_xgb))

print('\nClassification Report:')
print(classification_report(y_test, y_pred_xgb))

In [ ]:
#Tune max_depth
results = []

for depth in depths:

  xgb = XGBClassifier(
    n_estimators=100,
    max_depth=depth,
    random_state=42,
    objective = 'binary:logistic', # Tells XGBoost to do binary classification
    eval_metric='logloss'
  )

  scores = cross_val_score(
    xgb,
    X_train,
    y_train,
    cv=5,
    scoring = 'accuracy'
  )

  results.append({
    'max_depth': depth,
    'accuracy': scores.mean()
  })

results_df = pd.DataFrame(results)
results_df.sort_values(by='accuracy', ascending=False)

In [ ]:
#Tune n_estimators
n_estimators_values = [50, 100, 200, 300, 500]

results = []

for n in n_estimators_values:

  forest = XGBClassifier(
    n_estimators=n,
    max_depth=1,
    random_state=42,
    objective = 'binary:logistic', # Tells XGBoost to do binary classification
    eval_metric='logloss'
  )

  scores = cross_val_score(
    forest,
    X_train,
    y_train,
    cv=5,
    scoring = 'accuracy'
  )

  results.append({
    'n_estimators': n,
    'accuracy': scores.mean()
  })

results_df = pd.DataFrame(results)
results_df.sort_values(by='accuracy', ascending=False)

In [ ]:
#Tune learning_rate
learning_rates = [0.01, 0.05, 0.1, 0.2]

results = []

for n in learning_rates:

  xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=n,
    max_depth=1,
    random_state=42,
    objective = 'reg:squarederror'
  )

  scores = cross_val_score(
    xgb,
    X_train,
    y_train,
    cv=5,
    scoring = 'accuracy'
  )

  rmse_scores = -scores

  results.append({
    'learning_rate': n,
    'accuracy': scores.mean()
  })

results_df = pd.DataFrame(results)
results_df.sort_values(by='accuracy', ascending=False)

In [ ]:
#Final XGBoost model
xgb = XGBClassifier(
  n_estimators=300,   #Boosting rounds/trees
  learning_rate=0.20, #Controls how much each new tree contributest to model
  max_depth=1,        #Max deptho or each tree
  random_state=42,
  objective = 'binary:logistic', # Tells XGBoost to do binary classification
  eval_metric='logloss'
)

#Fit
xgb.fit(X_train, y_train)

#Predict
y_pred_xgb = xgb.predict(X_test)

#Probability
y_prob_xgb = xgb.predict_proba(X_test)[:,1]

print('Accuracy:', accuracy_score(y_test, y_pred_xgb))
print('Precision:', precision_score(y_test, y_pred_xgb))
print('Recall:', recall_score(y_test, y_pred_xgb))
print('F1:', f1_score(y_test, y_pred_xgb))
print('ROC-AUC:', roc_auc_score(y_test, y_prob_xgb))

print('\nClassification Report:')
print(classification_report(y_test, y_pred_xgb))

In [ ]:
#Feature importance
xgb_importances = pd.Series(
  xgb.feature_importances_,
  index=features
).sort_values(ascending=False)

xgb_importances.head(10)

##Data Visualization

In [ ]:
display(
  graphviz.Source(
    tree.export_graphviz(
      dt,
      feature_names = X.columns,
      filled = True,
    )))

In [ ]:
dt_importances.head(10).sort_values().plot(
  kind='barh',
  title='Decision Tree Feature Importance'
)

In [ ]:
#Feature importance
forest_importances = pd.Series(
  rf.feature_importances_,
  index=features
).sort_values(ascending=False)

#Plot
forest_importances.head(10).sort_values().plot(
  kind='barh',
  title='Random Forest Feature Importance'
)

In [ ]:
xgb_importances.head(10).sort_values().plot(
  kind='barh',
  title='XGBoost Feature Importance'
)

In [ ]:
results = pd.DataFrame({
  'Model': ['Decision Tree', 'Random Forest', 'XGBoost'],

  'Accuracy': [
    accuracy_score(y_test, y_pred_dt),
    accuracy_score(y_test, y_pred_rf),
    accuracy_score(y_test, y_pred_xgb)
  ],

  'F1': [
    f1_score(y_test, y_pred_dt),
    f1_score(y_test, y_pred_rf),
    f1_score(y_test, y_pred_xgb)
  ],

  'ROC-AUC': [
    roc_auc_score(y_test, y_prob_dt),
    roc_auc_score(y_test, y_prob_rf),
    roc_auc_score(y_test, y_prob_xgb)
  ]
})

results

In [ ]:
#Melt df to restructure for Seaborn
results_melted = results.melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(10, 6))
sns.barplot(
    data=results_melted,
    x='Metric',
    y='Score',
    hue='Model',
    palette=['#0097A7', '#FF9F24',  '#6366F1']
)

plt.title('Performance Metrics by Processing Method', fontsize=14)
plt.ylabel('Score', fontsize=12)
plt.xlabel('Metric', fontsize=12)
plt.ylim(0, 1.1) #Headroom

#Move legend
plt.legend(title='Method', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

plt.savefig('evaluation.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

#Horizontal
sns.barplot(
    data=feature_importance.head(10),
    x='importance',
    y='feature',
    palette='viridis'
)

# Add titles and labels for clarity
plt.title('Top 10 Feature Importances - XGBoost Forest', fontsize=14)
plt.xlabel('Relative Importance', fontsize=12)
plt.ylabel('Features', fontsize=12)

# Adjust layout and display
plt.tight_layout()
plt.show()

##Communication of Results

In [ ]:
importance_comparison = pd.DataFrame({
  'Decision Tree': dt_importances,
  'Random Forest': forest_importances,
  'XGBoost': xgb_importances
})

importance_comparison